In [1]:
import pandas as pd
import glob
import re
import sys
from datetime import datetime
# must also have openpyxl and xlsxwriter installed

# Define function to handle data cleaning
def clean(df):
    df = df.copy()
    # Handle Datetimes: convert YYYYMMDD to string
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].dt.strftime('%Y%m%d')
    # Force to String and handle .0s, nans/nulls, leading/trailing whitespace, and line breaks 
    for col in df.columns:
        df[col] = (
            df[col].astype(str)
            .replace(r'\.0$', '', regex=True)
            .replace(['nan', 'NaN', 'None', 'NaT'], '')
            .str.strip()
            .replace(r'[\n\r\t]+', ' ', regex=True)
        )
        # Ensure ints like Grade or Zip Code have no leading zeroes, convert to a string
        df[col] = df[col].apply(lambda x: str(int(x)) if x.isdigit() else x) 
    return df

In [3]:
# User Choice Configuration
# 1: Compares the two most recent UTREx validation files
# 2: Compares the latest UTREx validation file against the previous 'New Warnings' report
print("--- Comparison Mode ---")
print("1: Fresh Run (Compare the two most recent UTREx validation files)")
print("2: Returning Run (Compare latest UTREx validation file against your previous 'New Warnings' report)")

while True:
    choice = input("Select mode (1 or 2): ").strip()
    if choice in ['1', '2']:
        run_mode = int(choice)
        break
    print("Please enter 1 or 2.")

raw_files = sorted(glob.glob("UTREx-Level1AndLevel2-Validations-*.xlsx"))

# Safety check for empty list
if not raw_files:
    print("Error: No UTREx validation files found in the folder.")
    sys.exit()

# Sort raw UTREx validation files by collection 
try:
    raw_files.sort(key=lambda f: int(re.findall(r'\d+', f)[-1]) if re.findall(r'\d+', f) else 0)
except IndexError:
    print("Error: One of your filenames doesn't have a number at the end.")
    sys.exit()

# Check that files are there and able to be compared
if run_mode == 1:
    if len(raw_files) < 2:
        print(f"Error: Found only {len(raw_files)} file(s). Need 2 for Mode 1.")
        sys.exit()
    old_path = raw_files[-2]
    new_path = raw_files[-1]
    print(f"Mode 1: Comparing {old_path} vs {new_path}")

elif run_mode == 2:
    report_files = glob.glob("New_Warnings_Report_[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9].xlsx")
    if not report_files:
        print("Error: No previous 'New_Warnings_Report_YYYYMMDD.xlsx' report found.")
        sys.exit()
    report_files.sort()
    old_path = report_files[-1]
    new_path = raw_files[-1]
    print(f"Mode 2: Comparing previous report ({old_path}) against latest raw data ({new_path})")

# Read in selected Excel files    
old_excel = pd.ExcelFile(old_path)
new_excel = pd.ExcelFile(new_path)

# Initiate output filename
today_str = datetime.now().strftime('%Y%m%d')
output_filename = f"New_Warnings_Report_{today_str}.xlsx"

# Use ExcelWriter to save multiple sheets to one file
with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    
    # Get sheets in new file
    new_sheets = new_excel.sheet_names    
    for sheet in new_sheets:
        # Load the new data
        df_new = pd.read_excel(new_excel, sheet_name=sheet)
        
        # Preserve order of columns in file
        original_cols = [c for c in df_new.columns if c not in ['CollectTS', 'DateRecorded']]

        # Check if each sheet existed in the old file
        if sheet in old_excel.sheet_names:
            df_old = pd.read_excel(old_excel, sheet_name=sheet)
        else:
            # Create an empty DataFrame with the same columns so the merge doesn't crash
            df_old = pd.DataFrame(columns=df_new.columns)
        
        # Apply cleaning to dfs
        df_old = clean(df_old)
        df_new = clean(df_new)

        # Drop any columns titles CollectTS to ignore the timestamp of the collection 
        df_old = df_old.drop(columns=['CollectTS','DateRecorded'], errors='ignore')
        df_new = df_new.drop(columns=['CollectTS','DateRecorded'], errors='ignore')
        
        # For any columns that change by the day (membership, attendance, etc.), overwrite yesterday's with today's data
        # Use the following columns as keys to match on
        id_cols = ['ErrorMessage','StudentNumber','LEANumber','LeaNumber','SchoolNumber','EntryDate','ScramEntryDate','YICEntryDate','StatewideStudentID','IncidentID','CourseRecordID','otherLEANumber','otherSchoolNumber','otherEntryDate','otherScramEntryDate']
        actual_ids = [col for col in id_cols if col in df_new.columns]

        # Set the index using only the columns that exist on this tab
        for col in actual_ids:
            df_old[col] = df_old[col].astype(str).str.strip()
            df_new[col] = df_new[col].astype(str).str.strip()
        
        df_old.set_index(actual_ids, inplace=True)
        df_new.set_index(actual_ids, inplace=True)
        df_old.sort_index(inplace=True)
        df_new.sort_index(inplace=True)
        
        # Replace old file's ExtendedDescription for warnings that have values changing daily to new file's description
        mask = df_new['ErrorCode'].astype(str).isin(['S1.383w','S1.329w','04','4'])       
        updates = df_new.loc[mask, ['ExtendedDescription']]
        df_old.update(updates)
        
        # Replace old file's "counting" values with new file's values (we want to ignore these changes)
        noisy_cols = ['SchoolMembership','DaysAttended','ExcusedAbsences','UnexcusedAbsences','AbsencesDueToSuspension','CumulativeGPA','ScramMembership','YICMembership','YICMembershipActual','ActualMembership']
        existing_noisy_cols = [c for c in noisy_cols if c in df_new.columns]
        df_old.update(df_new[existing_noisy_cols])

        # Address datetime, type mismatch, NaN issues, convert all columns to strings to prevent errors
        # Handle Datetimes
        for df in [df_old, df_new]:
            for col in df.columns:
                if pd.api.types.is_datetime64_any_dtype(df[col]):
                    df[col] = df[col].dt.strftime('%Y%m%d')
        
        # Handle the different column names for filtering
        col_name = "Severity" if sheet == "Level 2 Errors" else "SeverityCode"
        
        if col_name not in df_old.columns or col_name not in df_new.columns:
            print(f"Skipping {sheet}: Column '{col_name}' missing.")
            continue
        
        # Filter for only warning validations and drop any duplicate rows
        # Filter new warnings
        if col_name in df_new.columns:
            df_new_warn = df_new[df_new[col_name] == "Warning"].drop_duplicates()
            
            # Filter old warnings
            if col_name in df_old.columns:
                df_old_warn = df_old[df_old[col_name] == "Warning"].drop_duplicates()
            else:
                df_old_warn = pd.DataFrame(columns=df_new.columns)
 
            # Find new warnings
            comparison = df_old_warn.reset_index().merge(df_new_warn.reset_index(), how='outer', indicator=True)
            new_warnings = comparison[comparison['_merge'] == 'right_only'].drop(columns=['_merge'])

            # Save to the new Excel file if any new warnings exist
            if not new_warnings.empty:
                final_cols = [c for c in original_cols if c in new_warnings.columns]
                new_warnings = new_warnings[final_cols]
                
                print(f"Adding {len(new_warnings)} new warnings from '{sheet}' to report.")
                new_warnings.to_excel(writer, sheet_name=sheet, index=False)
            else:
                print(f"No new warnings in '{sheet}'.")

print(f"\nDone! Report saved as: {output_filename}")

--- Comparison Mode ---
1: Fresh Run (Compare the two most recent UTREx validation files)
2: Returning Run (Compare latest UTREx validation file against your previous 'New Warnings' report)
Mode 2: Comparing previous report (New_Warnings_Report_20260416.xlsx) against latest raw data (UTREx-Level1AndLevel2-Validations-99-772389.xlsx)
Adding 9 new warnings from 'Student' to report.
Adding 1 new warnings from 'Free And Reduced' to report.
No new warnings in 'Incident'.
Adding 1 new warnings from 'Student List' to report.
Adding 4 new warnings from 'Student Transcript' to report.
Adding 2 new warnings from 'Level 2 Errors' to report.

Done! Report saved as: New_Warnings_Report_20260417.xlsx
